# Notebook 04 — 풀 파이프라인 통합
## TechDocRAG: FastAPI 서빙 + Streamlit UI + Docker
### 이 노트북에서 배울 것
- FastAPI 서버 코드 작성 (POST /ingest, POST /query, GET /health)
- Streamlit 데모 UI (PDF 업로드 → 질문 → 답변 + 출처 이미지)
- E2E 파이프라인 벤치마크
- Docker 빌드 구성

In [ ]:
# 한글 폰트 + 기본 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import json, time
from pathlib import Path

ROOT = Path().absolute().parent
print(f"프로젝트 루트: {ROOT}")
print(f"app/ 경로: {ROOT / 'app'}")

## 1. FastAPI 서버 코드 생성
`app/` 폴더에 모듈 파일들을 직접 작성합니다.

In [ ]:
# app/schemas.py 생성
schemas_code = '''from pydantic import BaseModel
from typing import Optional

class QueryRequest(BaseModel):
    question: str
    top_k: int = 2

class SourceInfo(BaseModel):
    chunk_id: str
    page_num: int
    score: float

class QueryResponse(BaseModel):
    answer: str
    sources: list[SourceInfo]
    model_used: str
    elapsed_sec: float

class IngestResponse(BaseModel):
    doc_name: str
    pages_indexed: int
    elapsed_sec: float

class HealthResponse(BaseModel):
    status: str
    indexed_docs: int
    ollama_available: bool
    version: str
'''

app_dir = ROOT / 'app'
app_dir.mkdir(exist_ok=True)
(app_dir / 'schemas.py').write_text(schemas_code, encoding='utf-8')
print("app/schemas.py 생성 완료")

In [ ]:
# app/ingestion.py 생성
ingestion_code = r'''"""PDF 인제스천: 이미지 변환 + 텍스트/표 추출 + 청킹"""
import fitz, pdfplumber, re, json
from pathlib import Path
from dataclasses import dataclass

OUTPUT_DIR = Path(__file__).parent.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class PageChunk:
    doc_name: str
    page_num: int
    chunk_id: str
    text: str
    image_path: str
    content_type: str
    has_table: bool
    char_count: int

def pdf_to_images(pdf_path: Path, dpi: int = 200) -> list[Path]:
    doc = fitz.open(str(pdf_path))
    img_dir = OUTPUT_DIR / pdf_path.stem
    img_dir.mkdir(parents=True, exist_ok=True)
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []
    for i, page in enumerate(doc):
        p = img_dir / f"page_{i:03d}.png"
        page.get_pixmap(matrix=mat, alpha=False).save(str(p))
        paths.append(p)
    doc.close()
    return paths

def _table_to_markdown(table):
    if not table or not table[0]:
        return ""
    rows = [[str(c).strip() if c else "" for c in row] for row in table]
    widths = [max(len(rows[r][c]) for r in range(len(rows))) for c in range(len(rows[0]))]
    lines = []
    for i, row in enumerate(rows):
        lines.append("| " + " | ".join(cell.ljust(widths[c]) for c, cell in enumerate(row)) + " |")
        if i == 0:
            lines.append("| " + " | ".join("-" * widths[c] for c in range(len(row))) + " |")
    return "\n".join(lines)

def ingest_pdf(pdf_path: Path) -> list[PageChunk]:
    image_paths = pdf_to_images(pdf_path)
    doc = fitz.open(str(pdf_path))

    # 텍스트 블록 추출
    all_blocks = []
    for page_num, page in enumerate(doc):
        for block in page.get_text("dict")["blocks"]:
            if block.get("type") != 0:
                continue
            for line in block.get("lines", []):
                texts, sizes = [], []
                for span in line.get("spans", []):
                    t = span["text"].strip()
                    if t:
                        texts.append(t)
                        sizes.append(span["size"])
                if texts:
                    all_blocks.append((page_num, " ".join(texts), sum(sizes)/len(sizes)))

    mean_size = sum(b[2] for b in all_blocks) / len(all_blocks) if all_blocks else 11

    # 페이지별 그룹
    page_texts: dict[int, list] = {}
    for page_num, text, size in all_blocks:
        page_texts.setdefault(page_num, []).append(
            f"## {text}" if size > mean_size * 1.2 else text
        )

    # 표 추출
    page_tables: dict[int, list] = {}
    with pdfplumber.open(str(pdf_path)) as pdf:
        for i, page in enumerate(pdf.pages):
            mds = [_table_to_markdown(t) for t in page.extract_tables() if t]
            if mds:
                page_tables[i] = mds

    doc.close()

    chunks = []
    for page_num in range(len(image_paths)):
        parts = page_texts.get(page_num, [])
        page_text = "\n".join(parts)
        t_mds = page_tables.get(page_num, [])
        if t_mds:
            page_text += "\n\n" + "\n\n".join(t_mds)

        tlen = sum(len(m) for m in t_mds)
        ctype = "table_heavy" if tlen > len(page_text)*0.5 else ("text_heavy" if len(page_text) > 200 else "mixed")

        chunks.append(PageChunk(
            doc_name=pdf_path.stem,
            page_num=page_num,
            chunk_id=f"{pdf_path.stem}_p{page_num:03d}",
            text=page_text.strip(),
            image_path=str(image_paths[page_num]),
            content_type=ctype,
            has_table=len(t_mds) > 0,
            char_count=len(page_text)
        ))
    return chunks
'''

(app_dir / 'ingestion.py').write_text(ingestion_code, encoding='utf-8')
print("app/ingestion.py 생성 완료")

In [ ]:
# app/retriever.py 생성
retriever_code = r'''"""하이브리드 검색 + Qwen2.5-VL 답변 생성"""
import re, pickle, base64, urllib.request
from pathlib import Path
from dataclasses import dataclass

import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

ROOT       = Path(__file__).parent.parent
VECTOR_DIR = ROOT / "vector_db"
OUTPUT_DIR = ROOT / "output"
MODEL_DIR  = ROOT / "models"

# ── 싱글톤 ────────────────────────────────────────
_embedder   = None
_collection = None
_bm25_data  = None

def _get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer('BAAI/bge-m3', cache_folder=str(MODEL_DIR))
    return _embedder

def _get_collection():
    global _collection
    if _collection is None:
        client = chromadb.PersistentClient(path=str(VECTOR_DIR))
        _collection = client.get_collection("tech_docs")
    return _collection

def _get_bm25():
    global _bm25_data
    if _bm25_data is None:
        p = OUTPUT_DIR / "bm25_index.pkl"
        if p.exists():
            with open(p, "rb") as f:
                _bm25_data = pickle.load(f)
    return _bm25_data


@dataclass
class RetrievedChunk:
    chunk_id: str; text: str; image_path: str
    page_num: int; doc_name: str; content_type: str
    hybrid_score: float; dense_score: float; bm25_score: float


def _tok(text: str) -> list[str]:
    return re.findall(r'[a-z0-9]+|[가-힣]{2,}', text.lower())


def search(query: str, top_k: int = 2) -> list[RetrievedChunk]:
    emb = _get_embedder()
    col = _get_collection()
    bd  = _get_bm25()

    q_emb = emb.encode([query], normalize_embeddings=True)
    dr    = col.query(query_embeddings=q_emb.tolist(),
                      n_results=min(top_k*2, col.count()),
                      include=["documents","metadatas","distances"])
    dense = {dr["ids"][0][i]: 1 - dr["distances"][0][i]
             for i in range(len(dr["ids"][0]))}

    bm25_scores = {}
    if bd:
        raw  = bd["bm25"].get_scores(_tok(query))
        mx   = max(raw) if max(raw) > 0 else 1
        bm25_scores = {bd["chunk_ids"][i]: float(raw[i])/mx
                       for i in range(len(bd["chunk_ids"]))}

    all_ids = set(dense) | set(k for k, v in bm25_scores.items() if v > 0)
    hybrid  = {cid: 0.7*dense.get(cid,0) + 0.3*bm25_scores.get(cid,0)
               for cid in all_ids}
    ranked  = sorted(hybrid.items(), key=lambda x: x[1], reverse=True)[:top_k]

    # 메타데이터 조회
    ids    = [r[0] for r in ranked]
    meta_r = col.get(ids=ids, include=["metadatas","documents"])
    meta_m = {meta_r["ids"][i]: meta_r["metadatas"][i] for i in range(len(meta_r["ids"]))}
    doc_m  = {meta_r["ids"][i]: meta_r["documents"][i] for i in range(len(meta_r["ids"]))}

    results = []
    for cid, h in ranked:
        m = meta_m.get(cid, {})
        results.append(RetrievedChunk(
            chunk_id=cid, text=doc_m.get(cid,""),
            image_path=m.get("image_path",""), page_num=m.get("page_num",0),
            doc_name=m.get("doc_name",""), content_type=m.get("content_type",""),
            hybrid_score=round(h,4), dense_score=round(dense.get(cid,0),4),
            bm25_score=round(bm25_scores.get(cid,0),4)
        ))
    return results


def _check_ollama() -> bool:
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        return True
    except:
        return False


def generate_answer(question: str, results: list[RetrievedChunk],
                    ollama_model: str = "qwen2.5vl:7b") -> tuple[str, str]:
    context = "\n\n---\n\n".join(
        f"[문서 {i+1}: {r.doc_name} / 페이지 {r.page_num+1}]\n{r.text}"
        for i, r in enumerate(results)
    )
    fallback = {
        "no_result": "관련 문서를 찾을 수 없습니다.",
    }
    if not results:
        return fallback["no_result"], "rule-based"

    if not _check_ollama():
        top = results[0]
        return (f"[{top.doc_name} / 페이지 {top.page_num+1}] 관련 내용:\n\n"
                f"{top.text[:600]}\n\n(※ Ollama 미연결)"), "rule-based"

    try:
        import ollama
        imgs = []
        for r in results[:2]:
            if r.image_path and Path(r.image_path).exists():
                with open(r.image_path, "rb") as f:
                    imgs.append(base64.b64encode(f.read()).decode())

        msg = {
            "role": "user",
            "content": (f"다음 문서를 참고하여 질문에 답하세요.\n\n"
                        f"[참고 문서]\n{context}\n\n[질문]\n{question}\n\n[답변]")
        }
        if imgs:
            msg["images"] = imgs

        resp = ollama.chat(
            model=ollama_model,
            messages=[
                {"role":"system","content":"당신은 EV 기술 문서 전문가입니다. 제공된 문서와 이미지를 기반으로 정확하게 답하세요."},
                msg
            ],
            options={"temperature": 0.1}
        )
        return resp["message"]["content"], ollama_model
    except Exception as e:
        top = results[0]
        return (f"[{top.doc_name} / 페이지 {top.page_num+1}]\n{top.text[:600]}"
                f"\n\n(Ollama 오류: {e})"), "rule-based"
'''

(app_dir / 'retriever.py').write_text(retriever_code, encoding='utf-8')
print("app/retriever.py 생성 완료")

In [ ]:
# app/main.py 생성 — FastAPI 엔트리포인트
main_code = r'''"""TechDocRAG FastAPI 서버"""
import time, shutil, tempfile
from pathlib import Path

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from schemas import QueryRequest, QueryResponse, SourceInfo, IngestResponse, HealthResponse
import ingestion, retriever as ret

app = FastAPI(
    title="TechDocRAG API",
    description="멀티모달 기술 문서 RAG — PDF(텍스트+이미지+표) 검색 + Qwen2.5-VL 답변",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware, allow_origins=["*"],
    allow_methods=["*"], allow_headers=["*"]
)


@app.get("/health", response_model=HealthResponse, tags=["Health"])
def health():
    """서버 상태 및 인덱스 현황 확인"""
    try:
        col   = ret._get_collection()
        count = col.count()
    except:
        count = 0
    return HealthResponse(
        status="healthy",
        indexed_docs=count,
        ollama_available=ret._check_ollama(),
        version="1.0.0"
    )


@app.post("/ingest", response_model=IngestResponse, tags=["Ingestion"])
async def ingest(file: UploadFile = File(...)):
    """
    PDF 파일을 업로드하여 인덱싱합니다.
    - 페이지 이미지 변환 (200 DPI)
    - 텍스트 + 표 추출
    - BGE-M3 임베딩 → ChromaDB 저장
    """
    if not file.filename.endswith(".pdf"):
        raise HTTPException(status_code=400, detail="PDF 파일만 지원합니다.")

    t0 = time.time()

    # 임시 저장
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        shutil.copyfileobj(file.file, tmp)
        tmp_path = Path(tmp.name)

    try:
        chunks = ingestion.ingest_pdf(tmp_path)
    finally:
        tmp_path.unlink(missing_ok=True)

    if not chunks:
        raise HTTPException(status_code=422, detail="PDF에서 텍스트를 추출할 수 없습니다.")

    # 임베딩 + ChromaDB 저장
    from sentence_transformers import SentenceTransformer
    import chromadb

    ROOT       = Path(__file__).parent.parent
    embedder   = ret._get_embedder()
    client     = chromadb.PersistentClient(path=str(ROOT / "vector_db"))

    # 기존 문서 덮어쓰기
    try:
        col = client.get_collection("tech_docs")
    except:
        col = client.create_collection("tech_docs", metadata={"hnsw:space":"cosine"})

    texts = [c.text for c in chunks]
    embs  = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

    col.upsert(
        ids        = [c.chunk_id for c in chunks],
        embeddings = embs.tolist(),
        documents  = texts,
        metadatas  = [{
            "doc_name":     c.doc_name,
            "page_num":     c.page_num,
            "image_path":   c.image_path,
            "content_type": c.content_type,
            "has_table":    str(c.has_table),
            "char_count":   c.char_count
        } for c in chunks]
    )

    # BM25 업데이트
    import re, pickle
    from rank_bm25 import BM25Okapi

    def tok(t): return re.findall(r'[a-z0-9]+|[가-힣]{2,}', t.lower())
    corpus = [tok(c.text) for c in chunks]
    bm25   = BM25Okapi(corpus)
    bm25_save = {"bm25": bm25, "tokenized_corpus": corpus,
                 "chunk_ids": [c.chunk_id for c in chunks]}
    with open(ROOT / "output" / "bm25_index.pkl", "wb") as f:
        pickle.dump(bm25_save, f)

    # 싱글톤 리셋 (새 인덱스 반영)
    ret._collection = None
    ret._bm25_data  = None

    return IngestResponse(
        doc_name=chunks[0].doc_name,
        pages_indexed=len(chunks),
        elapsed_sec=round(time.time() - t0, 2)
    )


@app.post("/query", response_model=QueryResponse, tags=["RAG"])
def query(req: QueryRequest):
    """
    질문을 받아 관련 문서를 검색하고 Qwen2.5-VL로 답변을 생성합니다.
    - 하이브리드 검색 (BGE-M3 70% + BM25 30%)
    - 페이지 이미지를 Vision LLM에 주입
    - Ollama 미연결 시 Rule-based 폴백
    """
    results = ret.search(req.question, top_k=req.top_k)
    if not results:
        raise HTTPException(status_code=404, detail="관련 문서를 찾을 수 없습니다.")

    t0 = time.time()
    answer, model_used = ret.generate_answer(req.question, results)

    return QueryResponse(
        answer=answer,
        sources=[SourceInfo(chunk_id=r.chunk_id, page_num=r.page_num,
                            score=r.hybrid_score) for r in results],
        model_used=model_used,
        elapsed_sec=round(time.time() - t0, 2)
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

(app_dir / 'main.py').write_text(main_code, encoding='utf-8')
(app_dir / '__init__.py').write_text('', encoding='utf-8')
print("app/main.py 생성 완료")
print("app/__init__.py 생성 완료")

## 2. Streamlit UI 생성

In [ ]:
# app/streamlit_app.py 생성
streamlit_code = r'''"""TechDocRAG Streamlit 데모 UI"""
import streamlit as st
import requests
from pathlib import Path

API_URL = "http://localhost:8000"

st.set_page_config(page_title="TechDocRAG", page_icon="📄", layout="wide")
st.title("📄 TechDocRAG — 기술 문서 멀티모달 RAG")
st.caption("PDF 문서(표·다이어그램 포함)를 업로드하고 자연어로 질문하세요.")

# 사이드바
with st.sidebar:
    st.header("⚙️ 설정")
    top_k = st.slider("검색 페이지 수 (top_k)", 1, 5, 2)
    st.divider()

    # 헬스체크
    try:
        h = requests.get(f"{API_URL}/health", timeout=3).json()
        st.success(f"서버 연결됨")
        st.metric("인덱싱된 페이지", h["indexed_docs"])
        st.metric("Ollama", "✓ 연결됨" if h["ollama_available"] else "✗ 폴백 모드")
    except:
        st.error("서버 연결 실패 — FastAPI 서버를 먼저 실행하세요")
        st.code("cd app && uvicorn main:app --port 8000")

    st.divider()

    # PDF 업로드
    st.header("📂 문서 업로드")
    pdf_file = st.file_uploader("PDF 파일 선택", type=["pdf"])
    if pdf_file and st.button("인덱싱 시작", use_container_width=True):
        with st.spinner("인덱싱 중..."):
            resp = requests.post(
                f"{API_URL}/ingest",
                files={"file": (pdf_file.name, pdf_file.getvalue(), "application/pdf")}
            )
        if resp.status_code == 200:
            d = resp.json()
            st.success(f"완료: {d['pages_indexed']}페이지 인덱싱 ({d['elapsed_sec']}초)")
        else:
            st.error(f"오류: {resp.text}")

# 메인 영역 — 질문 입력
st.header("💬 질문하기")
question = st.text_input(
    "질문을 입력하세요",
    placeholder="예: 배터리 팩의 공칭 전압은? / DTC 코드 P0A1E는 무엇인가요?"
)

if st.button("검색 + 답변 생성", type="primary", use_container_width=True) and question:
    with st.spinner("하이브리드 검색 + AI 답변 생성 중..."):
        try:
            resp = requests.post(
                f"{API_URL}/query",
                json={"question": question, "top_k": top_k},
                timeout=120
            )
        except requests.exceptions.ConnectionError:
            st.error("서버에 연결할 수 없습니다.")
            st.stop()

    if resp.status_code == 200:
        data = resp.json()

        # 답변
        st.subheader("🤖 AI 답변")
        st.markdown(data["answer"])
        st.caption(f"모델: `{data['model_used']}` | 응답시간: {data['elapsed_sec']}초")

        st.divider()

        # 출처 페이지 이미지
        st.subheader("📑 출처 페이지")
        cols = st.columns(len(data["sources"]))
        for col, src in zip(cols, data["sources"]):
            with col:
                st.markdown(f"**{src['chunk_id']}**  \n페이지 {src['page_num']+1} | score `{src['score']:.3f}`")
                img_path = Path(src["chunk_id"].replace("_p", "/page_").replace(
                    src["chunk_id"].split("_p")[0],
                    f"output/{src['chunk_id'].split('_p')[0]}"
                ).replace("/page_", f"/page_") )
                # 이미지 경로 직접 구성
                doc   = src["chunk_id"].split("_p")[0]
                pnum  = int(src["chunk_id"].split("_p")[1])
                ipath = Path(__file__).parent.parent / "output" / doc / f"page_{pnum:03d}.png"
                if ipath.exists():
                    st.image(str(ipath), use_container_width=True)
                else:
                    st.info("이미지를 찾을 수 없습니다.")
    elif resp.status_code == 404:
        st.warning("관련 문서를 찾을 수 없습니다. PDF를 먼저 업로드하고 인덱싱해주세요.")
    else:
        st.error(f"오류 {resp.status_code}: {resp.text}")

# 예시 질문
with st.expander("💡 예시 질문"):
    examples = [
        "배터리 팩의 공칭 전압과 용량은 얼마인가요?",
        "DTC 코드 P0A1E는 어떤 문제를 나타내나요?",
        "SOC는 어떻게 추정하나요?",
        "고전압 작업 시 안전 절차를 알려주세요.",
        "셀 밸런싱이란 무엇인가요?",
    ]
    for ex in examples:
        st.markdown(f"- {ex}")
'''

(app_dir / 'streamlit_app.py').write_text(streamlit_code, encoding='utf-8')
print("app/streamlit_app.py 생성 완료")

## 3. E2E 벤치마크

In [ ]:
# E2E 파이프라인 벤치마크 — NB01~NB03 모듈 재사용
import sys, json, time, re, pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path().absolute().parent
sys.path.insert(0, str(ROOT / 'app'))

# NB02/NB03 컴포넌트 재활용
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

with open(ROOT / 'output' / 'chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)
with open(ROOT / 'output' / 'bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)

embedder   = SentenceTransformer('BAAI/bge-m3', cache_folder=str(ROOT / 'models'))
collection = chromadb.PersistentClient(path=str(ROOT / 'vector_db')).get_collection("tech_docs")
bm25       = bm25_data['bm25']
chunk_ids  = bm25_data['chunk_ids']

def tok(text): return re.findall(r'[a-z0-9]+|[가-힣]{2,}', text.lower())

def hybrid_search(query, top_k=2):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    dr    = collection.query(query_embeddings=q_emb.tolist(),
                             n_results=min(top_k*2, collection.count()),
                             include=['distances'])
    dense = {dr['ids'][0][i]: 1-dr['distances'][0][i] for i in range(len(dr['ids'][0]))}
    raw   = bm25.get_scores(tok(query))
    mx    = max(raw) if max(raw) > 0 else 1
    bm25s = {chunk_ids[i]: float(raw[i])/mx for i in range(len(chunk_ids))}
    hybrid = {cid: 0.7*dense.get(cid,0)+0.3*bm25s.get(cid,0)
              for cid in set(dense)|set(k for k,v in bm25s.items() if v>0)}
    return sorted(hybrid.items(), key=lambda x: x[1], reverse=True)[:top_k]

# 5회 반복 벤치마크
bench_queries = [
    "배터리 팩 공칭 전압",
    "DTC 코드 P0A1E",
    "SOC 칼만 필터",
    "고전압 안전 절차",
    "셀 밸런싱",
]

times = []
print("벤치마크 실행 중 (5회 × 5 쿼리)...")
for _ in range(5):
    for q in bench_queries:
        t0 = time.time()
        hybrid_search(q, top_k=2)
        times.append((time.time() - t0) * 1000)

times_arr = np.array(times)
print(f"\n검색 응답시간 (25회 측정)")
print(f"  평균: {times_arr.mean():.1f} ms")
print(f"  P50:  {np.percentile(times_arr, 50):.1f} ms")
print(f"  P95:  {np.percentile(times_arr, 95):.1f} ms")
print(f"  최소: {times_arr.min():.1f} ms")
print(f"  최대: {times_arr.max():.1f} ms")

In [ ]:
# 벤치마크 결과 시각화
import matplotlib.pyplot as plt
import numpy as np

queries_label = ["공칭전압", "DTC코드", "SOC필터", "안전절차", "셀밸런싱"]
rounds = 5

# 각 쿼리별 평균 응답시간
per_query_times = []
for qi in range(len(bench_queries)):
    qt = [times[r * len(bench_queries) + qi] for r in range(rounds)]
    per_query_times.append(qt)

means   = [np.mean(qt) for qt in per_query_times]
p95s    = [np.percentile(qt, 95) for qt in per_query_times]
overall = [times_arr.mean(), np.percentile(times_arr, 50), np.percentile(times_arr, 95)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 쿼리별 응답시간 막대그래프
x = np.arange(len(queries_label))
axes[0].bar(x - 0.2, means, 0.4, label='평균', color='steelblue', alpha=0.85)
axes[0].bar(x + 0.2, p95s,  0.4, label='P95',  color='tomato',    alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(queries_label)
axes[0].set_ylabel('응답시간 (ms)')
axes[0].set_title('쿼리별 하이브리드 검색 응답시간')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 전체 통계 요약 막대그래프
labels_stat = ['평균', 'P50', 'P95']
colors_stat = ['steelblue', 'seagreen', 'tomato']
bars = axes[1].bar(labels_stat, overall, color=colors_stat, alpha=0.85, width=0.5)
for bar, val in zip(bars, overall):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}ms', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_ylabel('응답시간 (ms)')
axes[1].set_title('전체 검색 응답시간 요약 (25회 측정)')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, max(overall) * 1.3)

plt.suptitle('TechDocRAG 하이브리드 검색 벤치마크', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f"\n[결론] 평균 {times_arr.mean():.1f}ms — 실시간 검색 기준(200ms) 대비 {200/times_arr.mean():.1f}배 여유")

## 4. Docker 빌드 구성
`Dockerfile`과 `docker-compose.yml`을 생성합니다.

In [ ]:
# Dockerfile 생성
dockerfile = '''\
FROM python:3.11-slim

# 시스템 의존성 (PyMuPDF용 libmupdf, pdfplumber용 libpoppler)
RUN apt-get update && apt-get install -y --no-install-recommends \\
        libglib2.0-0 libgl1-mesa-glx libgomp1 \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /workspace

# 의존성 먼저 설치 (레이어 캐시 최적화)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 소스 코드 복사
COPY app/ ./app/
COPY output/ ./output/
COPY vector_db/ ./vector_db/

# ChromaDB, BM25, 임베딩 모델이 이미 빌드 시 포함됨
# (실운영에서는 볼륨 마운트 권장)

EXPOSE 8000

# FastAPI 서버 실행
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

(ROOT / 'Dockerfile').write_text(dockerfile, encoding='utf-8')
print("Dockerfile 생성 완료")
print()
print(dockerfile)

In [ ]:
# docker-compose.yml 생성
compose = '''\
version: "3.9"

services:
  api:
    build: .
    ports:
      - "8000:8000"
    volumes:
      # 인덱스 데이터를 컨테이너 재시작 시에도 유지
      - ./vector_db:/workspace/vector_db
      - ./output:/workspace/output
      - ./models:/workspace/models
    environment:
      - PYTHONUNBUFFERED=1
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 20s

  # Ollama Vision LLM (선택 — GPU 서버 환경에서 활성화)
  # ollama:
  #   image: ollama/ollama:latest
  #   ports:
  #     - "11434:11434"
  #   volumes:
  #     - ollama_data:/root/.ollama
  #   deploy:
  #     resources:
  #       reservations:
  #         devices:
  #           - driver: nvidia
  #             count: 1
  #             capabilities: [gpu]

# volumes:
#   ollama_data:
'''

(ROOT / 'docker-compose.yml').write_text(compose, encoding='utf-8')
print("docker-compose.yml 생성 완료")
print()
print(compose)

## 5. 프로젝트 완료 요약

In [ ]:
# 프로젝트 완료 요약 출력
from pathlib import Path
import json, pickle, os

ROOT = Path().absolute().parent

# 파일 구조 집계
nb_files   = list((ROOT / 'notebooks').glob('*.ipynb'))
app_files  = list((ROOT / 'app').glob('*.py'))
chunks_path = ROOT / 'output' / 'chunks.json'
bm25_path   = ROOT / 'output' / 'bm25_index.pkl'

n_chunks = 0
if chunks_path.exists():
    with open(chunks_path, 'r', encoding='utf-8') as f:
        n_chunks = len(json.load(f))

n_bm25 = 0
if bm25_path.exists():
    with open(bm25_path, 'rb') as f:
        n_bm25 = len(pickle.load(f)['chunk_ids'])

print("=" * 60)
print("  TechDocRAG — 프로젝트 완료 요약")
print("=" * 60)

print(f"""
[아키텍처]
  PDF (텍스트 + 표 + 다이어그램)
    ↓  PyMuPDF + pdfplumber
  PageChunk  →  BGE-M3 (1024-dim)  →  ChromaDB
                BM25Okapi          →  pkl 인덱스
    ↓  하이브리드 검색 (Dense 70% + BM25 30%)
  Top-K RetrievedChunk
    ↓  Qwen2.5-VL 7B (Ollama)  /  Rule-based 폴백
  구조화된 답변 + 출처 페이지 이미지

[인덱스 현황]
  ChromaDB 청크:  {n_chunks}개
  BM25 문서:      {n_bm25}개

[구현 파일]
  Notebooks: {len(nb_files)}개
    {chr(10).join('  - ' + f.name for f in sorted(nb_files))}
  App modules: {len(app_files)}개
    {chr(10).join('  - ' + f.name for f in sorted(app_files))}
  Dockerfile:        {'✓' if (ROOT / 'Dockerfile').exists() else '✗'}
  docker-compose.yml: {'✓' if (ROOT / 'docker-compose.yml').exists() else '✗'}

[API 엔드포인트]
  GET  /health        — 서버 상태 + 인덱스 현황
  POST /ingest        — PDF 업로드 → 자동 인덱싱
  POST /query         — 하이브리드 검색 + VL 답변 생성

[실행 방법]
  # 로컬 FastAPI
  cd app && uvicorn main:app --port 8000 --reload

  # Streamlit UI
  streamlit run app/streamlit_app.py

  # Docker
  docker-compose up --build
""")

print("=" * 60)
print("  Notebook 04 완료 — TechDocRAG 전체 구현 완성!")
print("=" * 60)